# Exact optimal transport on TFBind8

Load the same frozen GFNx autoregressive checkpoint as training. Enumerate every full string
$x \in \{A,C,G,T\}^8$ and compute its probability $p(x)$, without sampling.
The initial reward is $L_{\rm raw}(x)=p(x)$ and the terminal reward is
$R_{\rm raw}(x)=p(x)^\beta$. Balanced OT uses the normalized distributions

$$L(x)=\frac{p(x)}{Z_L},\qquad R(x)=\frac{p(x)^\beta}{Z_R}.$$

$Z_L$ is theoretically 1; we normalize the enumerated probabilities to remove floating-point
roundoff. $Z_R$ is the same terminal partition function computed by the trainer.

Each graph edge replaces one nucleotide and costs 1. The shortest-path distance is Hamming
distance, so this notebook computes exact, unregularized OT for that cost, up to the LP solver's
numerical tolerances. It retains the sparse graph-flow formulation from the permutation notebook.

`[BOS]`, `[EOS]`, and `[PAD]` belong to `NUCLEOTIDES_FULL_ALPHABET` and are handled by the
existing environment/sampler. They are not characters in the full strings of the cyclic graph.
There are $4^8=65,536$ states and $65,536\times 8\times 3=1,572,864$ directed replacement edges.

In [1]:
import sys
from pathlib import Path
from time import perf_counter

PROJECT_ROOT = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "envs/tfbind/tfbind.py").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import jax
import jax.numpy as jnp
import numpy as np
import scipy.sparse as sp
from scipy.optimize import linprog
from scipy.special import logsumexp

from envs.tfbind.tfbind import TFBind8Environment
from envs.tfbind.tokens import NUCLEOTIDES, NUCLEOTIDES_FULL_ALPHABET

CHECKPOINT = PROJECT_ROOT / "envs/tfbind/checkpoint/sampler.npz"
BETA = 2.0
REWARD_BATCH_SIZE = 1024
SOLVER_OPTIONS = {"disp": True}  # HiGHS options; optionally set time_limit in seconds.

np.set_printoptions(precision=8, suppress=True)

## Enumerate states and evaluate both rewards

State IDs use the same base-4 / NumPy flattening order as the TFBind environment and saved
training distributions. Only checkpoint evaluation uses JAX; normalization and the LP use
NumPy float64. Change `CHECKPOINT` and `BETA` above to match the experiment.

In [2]:
def enumerate_tfbind_states(max_length=8, nchar=len(NUCLEOTIDES)):
    """Full nucleotide strings in the environment's flattened distribution order."""
    state_ids = np.arange(nchar**max_length, dtype=np.int32)
    return np.stack(np.unravel_index(state_ids, (nchar,) * max_length), axis=-1)


def get_tfbind_distributions(env, batch_size=1024):
    states = enumerate_tfbind_states(env.max_length, env.nchar)

    @jax.jit
    def evaluate(batch):
        return env.log_initial_reward(batch), env.log_terminal_reward(batch)

    initial_chunks, terminal_chunks = [], []
    for start in range(0, len(states), batch_size):
        log_initial, log_terminal = evaluate(jnp.asarray(states[start:start + batch_size]))
        initial_chunks.append(np.asarray(log_initial, dtype=np.float64))
        terminal_chunks.append(np.asarray(log_terminal, dtype=np.float64))

    log_initial_rewards = np.concatenate(initial_chunks)
    log_terminal_rewards = np.concatenate(terminal_chunks)
    logZ_L = float(logsumexp(log_initial_rewards))
    logZ_R = float(logsumexp(log_terminal_rewards))
    L = np.exp(log_initial_rewards - logZ_L)
    R = np.exp(log_terminal_rewards - logZ_R)
    L /= L.sum()
    R /= R.sum()
    return {
        "states": states,
        "L": L,
        "R": R,
        "log_initial_rewards": log_initial_rewards,
        "log_terminal_rewards": log_terminal_rewards,
        "logZ_L": logZ_L,
        "logZ_R": logZ_R,
    }

In [3]:
env = TFBind8Environment(beta=BETA, checkpoint=str(CHECKPOINT))
distributions = get_tfbind_distributions(env, batch_size=REWARD_BATCH_SIZE)
states, L, R = (distributions[name] for name in ("states", "L", "R"))

print(f"Checkpoint: {CHECKPOINT}")
print(f"Alphabet: {NUCLEOTIDES}; full token alphabet: {NUCLEOTIDES_FULL_ALPHABET}")
print(f"States: {len(states):,}; sequence length: {env.max_length}; beta: {env.beta}")
print(f"log Z_L: {distributions['logZ_L']:.10f} (theoretically 0)")
print(f"log Z_R: {distributions['logZ_R']:.10f}")
print(f"Source mass: {L.sum():.12f}; target mass: {R.sum():.12f}")
print(f"TV(source, target): {0.5 * np.abs(L - R).sum():.8f}")

Checkpoint: /Users/kkorolev/mml/gfn_ot/envs/tfbind/checkpoint/sampler.npz
Alphabet: ['A', 'C', 'G', 'T']; full token alphabet: ['A', 'C', 'G', 'T', '[BOS]', '[EOS]', '[PAD]']
States: 65,536; sequence length: 8; beta: 2.0
log Z_L: -0.0000000304 (theoretically 0)
log Z_R: -10.2526855528
Source mass: 1.000000000000; target mass: 1.000000000000
TV(source, target): 0.30565146


## Build the action graph

As in `TFBind8Environment.step`, action `position * env.nchar + new_token` replaces that
position. The policy masks replacements by the current token, leaving 24 valid replacements
per state. Both directions of every replacement are present.

Action `env.stop_action == 32` stops at the current string. Its absorbed mass is already fixed
by $R(x)$ in the flow constraints below, so it does not need a graph edge. The OT objective
counts **replacements**. Adding one final stop action gives `objective + 1` for the minimum
expected action count including stopping, corresponding to the trainer's trajectory-length
convention when the terminal distribution is correct.

In [4]:
def build_tfbind_graph(max_length=8, nchar=len(NUCLEOTIDES)):
    """Unit-cost directed edges for every valid single-nucleotide replacement."""
    n_states = nchar**max_length
    degree = max_length * (nchar - 1)
    state_ids = np.arange(n_states, dtype=np.int32)
    heads = np.empty((n_states, degree), dtype=np.int32)

    for position in range(max_length):
        place_value = nchar ** (max_length - position - 1)
        current_token = (state_ids // place_value) % nchar
        for offset in range(1, nchar):
            new_token = (current_token + offset) % nchar
            edge = position * (nchar - 1) + offset - 1
            heads[:, edge] = state_ids + (new_token - current_token) * place_value

    tails = np.repeat(state_ids, degree)
    costs = np.ones(n_states * degree, dtype=np.float64)
    return sp.coo_matrix(
        (costs, (tails, heads.ravel())), shape=(n_states, n_states)
    ).tocsr()


graph = build_tfbind_graph(env.max_length, env.nchar)
assert graph.shape == (len(states), len(states))
assert np.all(np.diff(graph.indptr) == env.max_length * (env.nchar - 1))
print(f"Graph: {graph.shape[0]:,} states, {graph.nnz:,} directed edges")
print(f"Replacement cost: 1; stop action: {env.stop_action}")

Graph: 65,536 states, 1,572,864 directed edges
Replacement cost: 1; stop action: 32


## Solve sparse graph-flow OT

For edge flows $f_{uv}\geq 0$, solve

$$\min_f\sum_{(u,v)} f_{uv},\qquad
\sum_v f_{xv}-\sum_u f_{ux}=L(x)-R(x).$$

This is equivalent to Kantorovich OT with Hamming cost. It avoids the full
$65,536\times65,536$ distance matrix and coupling. The full LP still has about 1.57 million
variables, so the full solve is the expensive cell; HiGHS prints progress when `disp=True`.

The implementation rescales the flow variables by the number of states so that small
probabilities are represented more accurately relative to the solver's absolute tolerances.
Returned edge flows and the objective are converted back to probability units. No mass is
rounded to integers or removed. A time-limited or failed solve raises an error instead of
reporting its unfinished objective as the optimum.

In [5]:
def solve_hamming_ot_via_sparse_graph_flow(
    L,
    R,
    graph,
    drop_redundant_constraint=True,
    solver_options=None,
):
    """Exact Hamming OT on the directed TFBind replacement graph."""
    L = np.asarray(L, dtype=np.float64).reshape(-1)
    R = np.asarray(R, dtype=np.float64).reshape(-1)
    if graph.shape != (len(L), len(L)) or R.shape != L.shape:
        raise ValueError("L and R must contain one mass per graph state.")
    if not (np.isfinite(L).all() and np.isfinite(R).all()):
        raise ValueError("L and R must contain finite masses.")
    if np.any(L < 0) or np.any(R < 0):
        raise ValueError("L and R must be nonnegative.")
    if not np.isclose(L.sum(), R.sum(), atol=1e-12, rtol=1e-12):
        raise ValueError("L and R must have equal total mass; normalize the rewards first.")

    edges = graph.tocoo()
    n_states, n_edges = graph.shape[0], graph.nnz
    edge_ids = np.arange(n_edges, dtype=np.int32)
    rows = np.column_stack((edges.row, edges.col)).ravel()
    cols = np.repeat(edge_ids, 2)
    signs = np.tile(np.array([1.0, -1.0]), n_edges)
    incidence = sp.coo_matrix(
        (signs, (rows, cols)), shape=(n_states, n_edges)
    ).tocsr()

    flow_scale = float(n_states)
    balance = L - R
    A_eq = incidence[:-1] if drop_redundant_constraint else incidence
    b_eq = (balance[:-1] if drop_redundant_constraint else balance) * flow_scale
    options = {
        "primal_feasibility_tolerance": 1e-9,
        "dual_feasibility_tolerance": 1e-9,
    }
    options.update(solver_options or {})
    result = linprog(
        c=edges.data,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=(0, None),
        method="highs",
        options=options,
    )
    if not result.success:
        raise RuntimeError(f"Sparse graph-flow OT failed: {result.message}")

    flows = result.x / flow_scale
    max_balance_error = float(np.max(np.abs(incidence @ flows - balance)))
    if max_balance_error > 1e-8:
        raise RuntimeError(f"Flow conservation failed: max error {max_balance_error:.3g}")
    edge_flow = sp.coo_matrix(
        (flows, (edges.row, edges.col)), shape=graph.shape
    ).tocsr()
    edge_flow.eliminate_zeros()
    return {
        "edge_flow": edge_flow,
        "objective": float(result.fun / flow_scale),
        "max_balance_error": max_balance_error,
        "graph": graph,
        "flow_scale": flow_scale,
        "linprog_result": result,  # Raw HiGHS variables/objective use scaled mass units.
    }

## Check against a small dense OT problem

The dense reference is used only for 16 two-character strings. Check that sparse and dense
objectives agree, identical distributions cost zero, and moving all mass from `AA` to `TT`
costs exactly 2. The full calculation below always uses all 8 positions and all 65,536 states.

In [6]:
def solve_kantorovich_ot_lp(L, R, cost, drop_redundant_constraint=True):
    """Dense pairwise reference, only for the small validation problem."""
    L = np.asarray(L, dtype=np.float64)
    R = np.asarray(R, dtype=np.float64)
    cost = np.asarray(cost, dtype=np.float64)
    if cost.shape != (len(L), len(R)):
        raise ValueError("Cost matrix shape must be (len(L), len(R)).")
    if not np.isclose(L.sum(), R.sum(), atol=1e-12, rtol=1e-12):
        raise ValueError("L and R must have equal total mass.")
    sources, targets = np.where(np.isfinite(cost))
    edge_ids = np.arange(len(sources))
    A = sp.coo_matrix(
        (np.ones(2 * len(sources)),
         (np.concatenate((sources, len(L) + targets)), np.tile(edge_ids, 2))),
        shape=(len(L) + len(R), len(sources)),
    ).tocsr()
    b = np.concatenate((L, R))
    result = linprog(
        cost[sources, targets],
        A_eq=A[:-1] if drop_redundant_constraint else A,
        b_eq=b[:-1] if drop_redundant_constraint else b,
        bounds=(0, None),
        method="highs",
    )
    if not result.success:
        raise RuntimeError(f"Dense OT failed: {result.message}")
    coupling = np.zeros_like(cost)
    coupling[sources, targets] = result.x
    return {"coupling": coupling, "objective": float(result.fun)}


test_states = enumerate_tfbind_states(max_length=2)
test_graph = build_tfbind_graph(max_length=2)
rng = np.random.default_rng(0)
test_L, test_R = rng.dirichlet(np.ones(len(test_states)), size=2)
test_cost = np.count_nonzero(test_states[:, None, :] != test_states[None, :, :], axis=-1)
dense = solve_kantorovich_ot_lp(test_L, test_R, test_cost)
sparse = solve_hamming_ot_via_sparse_graph_flow(test_L, test_R, test_graph)
np.testing.assert_allclose(sparse["objective"], dense["objective"], atol=1e-9, rtol=0)

identity = solve_hamming_ot_via_sparse_graph_flow(test_L, test_L, test_graph)
assert identity["objective"] == 0
point_L, point_R = np.zeros(len(test_states)), np.zeros(len(test_states))
point_L[0], point_R[-1] = 1, 1
point = solve_hamming_ot_via_sparse_graph_flow(point_L, point_R, test_graph)
np.testing.assert_allclose(point["objective"], 2, atol=1e-9, rtol=0)
print(f"Checks passed: dense = sparse = {sparse['objective']:.10f}; identity = 0; AA -> TT = 2.")

Checks passed: dense = sparse = 0.7631478616; identity = 0; AA -> TT = 2.


## Full TFBind8 result

`result["edge_flow"][u, v]` is the optimal probability mass transported along replacement
edge `u -> v`; it is not a source-to-terminal coupling or a transition probability.

In [7]:
start = perf_counter()
result = solve_hamming_ot_via_sparse_graph_flow(L, R, graph, solver_options=SOLVER_OPTIONS)
elapsed = perf_counter() - start

print(f"Exact OT / minimum expected replacements: {result['objective']:.10f}")
print(f"Minimum expected action count including stop: {result['objective'] + 1:.10f}")
print(f"Maximum flow-conservation error: {result['max_balance_error']:.3e}")
print(f"Edges carrying positive flow: {result['edge_flow'].nnz:,}")
print(f"Solve time: {elapsed:.1f} s")

Running HiGHS 1.8.0 (git hash: 222cce7): Copyright (c) 2024 HiGHS under MIT licence terms
Coefficient ranges:
  Matrix [1e+00, 1e+00]
  Cost   [1e+00, 1e+00]
  Bound  [0e+00, 0e+00]
  RHS    [5e-05, 2e+01]
Presolving model
65535 rows, 1572864 cols, 3145680 nonzeros  0s
65535 rows, 1572864 cols, 3145680 nonzeros  0s
Presolve : Reductions: rows 65535(-0); columns 1572864(-0); elements 3145680(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 65535(40061.9) 1s
     110775     3.2739967660e+04 Pr: 11416(19998.4); Du: 0(1.96584e-08) 6s
     161207     3.3833961332e+04 Pr: 3438(8464.55); Du: 0(1.71724e-08) 11s
     174794     3.3857552713e+04 Pr: 2505(8263.11); Du: 0(2.22362e-08) 17s
     181027     3.3859962402e+04 Pr: 2123(6648.28); Du: 0(2.2965e-08) 22s
     185574     3.3880911265e+04 Pr: 1775(4836.19); Du: 0(1.85753e-08) 28s
     188787   

In [8]:
optimal_mean_length = result["objective"] + 1.0
optimal_edge_flow = result["edge_flow"]

optimal_mean_length

1.5172332142085083